In [1]:
"""
Cell 1 — Data loading + feature extraction (identical aggregation logic
to the SVM experiment, kept self-contained so this runs independently
in a fresh kernel session).
"""

import numpy as np
import pandas as pd
from pathlib import Path

DATASET_ROOT = Path("/kaggle/input/datasets/am2008/edaic-full-data/edaic_btp_dataset")
RAW_DIR = DATASET_ROOT / "raw_egemaps"
LABELS_DIR = DATASET_ROOT / "labels"

STATS = ["mean", "std", "median", "min", "max", "p25", "p75"]

def egemaps_path(pid):
    matches = list((RAW_DIR / f"{pid}_P").glob("*egemaps.csv"))
    return matches[0] if matches else None

def compute_participant_stats(arr):
    arr = np.where(np.isfinite(arr), arr, np.nan)
    stats = [
        np.nanmean(arr, axis=0), np.nanstd(arr, axis=0), np.nanmedian(arr, axis=0),
        np.nanmin(arr, axis=0), np.nanmax(arr, axis=0),
        np.nanpercentile(arr, 25, axis=0), np.nanpercentile(arr, 75, axis=0),
    ]
    vec = np.concatenate(stats)
    return np.where(np.isfinite(vec), vec, 0.0)

def load_split(split_name):
    labels = pd.read_csv(LABELS_DIR / f"{split_name}_split.csv")[["Participant_ID", "PHQ_Binary"]]
    rows, kept_pids, kept_labels = [], [], []
    feature_cols = None

    for _, row in labels.iterrows():
        pid, y = row["Participant_ID"], row["PHQ_Binary"]
        p = egemaps_path(pid)
        if p is None:
            print(f"[warn] missing raw file for participant {pid} ({split_name}) — skipped")
            continue
        d = pd.read_csv(p, sep=";")
        if feature_cols is None:
            feature_cols = [c for c in d.columns if c not in ("name", "frameTime")]
            assert len(feature_cols) == 23, f"Expected 23 feature columns, got {len(feature_cols)}"
        arr = d[feature_cols].values.astype(np.float64)
        rows.append(compute_participant_stats(arr))
        kept_pids.append(pid)
        kept_labels.append(y)

    feat_names = [f"{fc}_{s}" for s in STATS for fc in feature_cols]
    X = pd.DataFrame(rows, columns=feat_names)
    X.insert(0, "Participant_ID", kept_pids)
    X["PHQ_Binary"] = kept_labels
    return X

print("Loading train/dev/test raw eGeMAPS (275 files total)...")
train_data = load_split("train")
dev_data = load_split("dev")
test_data = load_split("test")

feature_cols = [c for c in train_data.columns if c not in ("Participant_ID", "PHQ_Binary")]
n_features = len(feature_cols)

print(f"\nFeature vector length: {n_features} (23 x 7 expected = 161)")
print(f"Train / Dev / Test counts: {len(train_data)} / {len(dev_data)} / {len(test_data)}")
print("Train class distribution:", train_data.PHQ_Binary.value_counts().to_dict())
print("Dev class distribution:", dev_data.PHQ_Binary.value_counts().to_dict())
print("Test class distribution:", test_data.PHQ_Binary.value_counts().to_dict())

Loading train/dev/test raw eGeMAPS (275 files total)...

Feature vector length: 161 (23 x 7 expected = 161)
Train / Dev / Test counts: 163 / 56 / 56
Train class distribution: {0: 126, 1: 37}
Dev class distribution: {0: 44, 1: 12}
Test class distribution: {0: 39, 1: 17}


In [2]:
"""
Cell 2 — Training. Impute (train-fit only, XGBoost handles NaN natively
but we impute anyway for consistency with the SVM pipeline and to keep
feature semantics comparable across experiments). Compact grid search
via StratifiedKFold on TRAIN only. scale_pos_weight computed from TRAIN only.
Early stopping uses a DEV-held eval set, never TEST.
"""

import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier

RANDOM_STATE = 42

X_train_raw = train_data[feature_cols].values
X_dev_raw = dev_data[feature_cols].values
X_test_raw = test_data[feature_cols].values
y_train = train_data["PHQ_Binary"].values
y_dev = dev_data["PHQ_Binary"].values
y_test = test_data["PHQ_Binary"].values

# Impute fitted on TRAIN only, transform-only on dev/test
imputer = SimpleImputer(strategy="median").fit(X_train_raw)
X_train = imputer.transform(X_train_raw)
X_dev = imputer.transform(X_dev_raw)
X_test = imputer.transform(X_test_raw)

# scale_pos_weight from TRAIN only
n_pos, n_neg = (y_train == 1).sum(), (y_train == 0).sum()
scale_pos_weight = n_neg / n_pos
print(f"scale_pos_weight (train-derived): {scale_pos_weight:.3f}  (neg={n_neg}, pos={n_pos})")

# ------------------------------------------------------------------
# Compact grid, kept small deliberately — n=163 train, high overfit risk
# ------------------------------------------------------------------
param_grid = [
    {"max_depth": 2, "n_estimators": 100, "learning_rate": 0.05, "subsample": 0.8, "colsample_bytree": 0.8},
    {"max_depth": 2, "n_estimators": 200, "learning_rate": 0.03, "subsample": 0.8, "colsample_bytree": 0.6},
    {"max_depth": 3, "n_estimators": 100, "learning_rate": 0.05, "subsample": 0.7, "colsample_bytree": 0.7},
    {"max_depth": 3, "n_estimators": 150, "learning_rate": 0.03, "subsample": 0.8, "colsample_bytree": 0.8},
    {"max_depth": 4, "n_estimators": 100, "learning_rate": 0.03, "subsample": 0.7, "colsample_bytree": 0.6},
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def cv_f1(params):
    """Mean F1 across StratifiedKFold on TRAIN only — no dev/test involved."""
    from sklearn.metrics import f1_score
    scores = []
    for train_idx, val_idx in cv.split(X_train, y_train):
        Xt, Xv = X_train[train_idx], X_train[val_idx]
        yt, yv = y_train[train_idx], y_train[val_idx]
        fold_pos_weight = (yt == 0).sum() / (yt == 1).sum()
        model = XGBClassifier(
            **params, scale_pos_weight=fold_pos_weight, random_state=RANDOM_STATE,
            eval_metric="logloss", use_label_encoder=False, verbosity=0,
        )
        model.fit(Xt, yt)
        preds = model.predict(Xv)
        scores.append(f1_score(yv, preds, zero_division=0))
    return np.mean(scores)

print("\nGrid search (train-only StratifiedKFold, scored by F1)...")
best_params, best_cv_f1 = None, -1
for params in param_grid:
    score = cv_f1(params)
    print(f"  {params} -> CV F1: {score:.4f}")
    if score > best_cv_f1:
        best_cv_f1, best_params = score, params

print(f"\nBest params: {best_params}")
print(f"Best train-CV F1: {best_cv_f1:.4f}")

# ------------------------------------------------------------------
# Final fit on full TRAIN, with early stopping monitored on DEV
# (dev is used to pick the stopping round, never test)
# ------------------------------------------------------------------
final_model = XGBClassifier(
    **best_params, scale_pos_weight=scale_pos_weight, random_state=RANDOM_STATE,
    eval_metric="logloss", use_label_encoder=False, verbosity=0,
    early_stopping_rounds=15,
)
final_model.fit(X_train, y_train, eval_set=[(X_dev, y_dev)], verbose=False)

print(f"\nFinal model trained. Best iteration (early stopping on DEV): "
      f"{final_model.best_iteration if hasattr(final_model, 'best_iteration') else 'N/A (no stopping triggered)'}")

scale_pos_weight (train-derived): 3.405  (neg=126, pos=37)

Grid search (train-only StratifiedKFold, scored by F1)...
  {'max_depth': 2, 'n_estimators': 100, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8} -> CV F1: 0.1967
  {'max_depth': 2, 'n_estimators': 200, 'learning_rate': 0.03, 'subsample': 0.8, 'colsample_bytree': 0.6} -> CV F1: 0.2252
  {'max_depth': 3, 'n_estimators': 100, 'learning_rate': 0.05, 'subsample': 0.7, 'colsample_bytree': 0.7} -> CV F1: 0.2106
  {'max_depth': 3, 'n_estimators': 150, 'learning_rate': 0.03, 'subsample': 0.8, 'colsample_bytree': 0.8} -> CV F1: 0.1571
  {'max_depth': 4, 'n_estimators': 100, 'learning_rate': 0.03, 'subsample': 0.7, 'colsample_bytree': 0.6} -> CV F1: 0.1661

Best params: {'max_depth': 2, 'n_estimators': 200, 'learning_rate': 0.03, 'subsample': 0.8, 'colsample_bytree': 0.6}
Best train-CV F1: 0.2252

Final model trained. Best iteration (early stopping on DEV): 36


In [3]:
"""
Cell 3 — DEV threshold selection. Sweep 0.10-0.90, pick by DEV F1 only.
TEST is not touched in this cell.
"""

import numpy as np
from sklearn.metrics import f1_score

dev_proba = final_model.predict_proba(X_dev)[:, 1]

thresholds = np.arange(0.10, 0.91, 0.01)
best_threshold, best_dev_f1 = 0.5, -1
for t in thresholds:
    preds = (dev_proba >= t).astype(int)
    f1 = f1_score(y_dev, preds, zero_division=0)
    if f1 > best_dev_f1:
        best_dev_f1, best_threshold = f1, t

print(f"Best DEV threshold: {best_threshold:.2f}  (DEV F1 at this threshold: {best_dev_f1:.4f})")

Best DEV threshold: 0.57  (DEV F1 at this threshold: 0.4211)


In [4]:
"""
Cell 4 — Final TEST evaluation. TEST is used exactly once, here, at the
threshold frozen from DEV in Cell 3.
"""

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, average_precision_score, confusion_matrix)

def compute_metrics(y_true, proba, threshold):
    preds = (proba >= threshold).astype(int)
    return {
        "Accuracy": accuracy_score(y_true, preds),
        "Precision": precision_score(y_true, preds, zero_division=0),
        "Recall": recall_score(y_true, preds, zero_division=0),
        "F1": f1_score(y_true, preds, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, proba),
        "PR_AUC": average_precision_score(y_true, proba),
        "ConfusionMatrix": confusion_matrix(y_true, preds).tolist(),
    }

test_proba = final_model.predict_proba(X_test)[:, 1]  # TEST touched for the first time, right here
dev_metrics = compute_metrics(y_dev, dev_proba, best_threshold)
test_metrics = compute_metrics(y_test, test_proba, best_threshold)

print("DEV  metrics:", {k: round(v, 4) if isinstance(v, float) else v for k, v in dev_metrics.items()})
print("TEST metrics:", {k: round(v, 4) if isinstance(v, float) else v for k, v in test_metrics.items()})

DEV  metrics: {'Accuracy': 0.8036, 'Precision': 0.5714, 'Recall': 0.3333, 'F1': 0.4211, 'ROC_AUC': np.float64(0.5095), 'PR_AUC': np.float64(0.3472), 'ConfusionMatrix': [[41, 3], [8, 4]]}
TEST metrics: {'Accuracy': 0.75, 'Precision': 0.5882, 'Recall': 0.5882, 'F1': 0.5882, 'ROC_AUC': np.float64(0.7466), 'PR_AUC': np.float64(0.6232), 'ConfusionMatrix': [[32, 7], [7, 10]]}


In [5]:
"""
Cell 5 — Feature importance (XGBoost gain-based), top 20 for the paper.
"""

import pandas as pd

importance = final_model.feature_importances_  # gain-based by default in sklearn API
importance_df = pd.DataFrame({"Feature": feature_cols, "Importance": importance})
importance_df = importance_df.sort_values("Importance", ascending=False).reset_index(drop=True)

print("Top 20 most important features (XGBoost, gain-based):")
print(importance_df.head(20).to_string(index=False))

importance_df.to_csv("/kaggle/working/xgb_feature_importance.csv", index=False)

Top 20 most important features (XGBoost, gain-based):
                  Feature  Importance
        Loudness_sma3_min    0.032821
        mfcc3_sma3_median    0.026973
   slope500-1500_sma3_p25    0.026253
        mfcc1_sma3_median    0.025673
    spectralFlux_sma3_p25    0.022245
           mfcc2_sma3_std    0.022113
slope500-1500_sma3_median    0.021902
   F3frequency_sma3nz_p25    0.021196
   F1frequency_sma3nz_p25    0.021136
 spectralFlux_sma3_median    0.020647
   slope500-1500_sma3_max    0.020515
           mfcc1_sma3_max    0.020415
           mfcc2_sma3_p25    0.019919
           mfcc1_sma3_p25    0.019828
           mfcc4_sma3_p25    0.019750
           mfcc3_sma3_min    0.019543
  F3frequency_sma3nz_mean    0.019205
   F1bandwidth_sma3nz_max    0.018866
           mfcc1_sma3_std    0.018434
          mfcc4_sma3_mean    0.018267


In [6]:
"""
Cell 6 — Compact experiment summary for the BTP results table.
"""

import pandas as pd

print("=" * 70)
print("EXPERIMENT SUMMARY — XGBoost on 7-stat aggregated eGeMAPS")
print("=" * 70)
print(f"Feature vector length     : {n_features}  (23 eGeMAPS features x 7 stats)")
print(f"Train / Dev / Test counts : {len(train_data)} / {len(dev_data)} / {len(test_data)}")
print(f"Train class distribution  : {train_data.PHQ_Binary.value_counts().to_dict()}")
print(f"Dev class distribution    : {dev_data.PHQ_Binary.value_counts().to_dict()}")
print(f"Test class distribution   : {test_data.PHQ_Binary.value_counts().to_dict()}")
print(f"scale_pos_weight (train)  : {scale_pos_weight:.3f}")
print(f"Best hyperparameters      : {best_params}")
print(f"Best DEV threshold        : {best_threshold:.2f}")
print("-" * 70)

summary_df = pd.DataFrame([
    {"Split": "DEV", **{k: v for k, v in dev_metrics.items() if k != "ConfusionMatrix"}},
    {"Split": "TEST", **{k: v for k, v in test_metrics.items() if k != "ConfusionMatrix"}},
])
print(summary_df.round(4).to_string(index=False))
print("-" * 70)
print("DEV  confusion matrix [[TN FP] [FN TP]]:", dev_metrics["ConfusionMatrix"])
print("TEST confusion matrix [[TN FP] [FN TP]]:", test_metrics["ConfusionMatrix"])
print("=" * 70)

summary_df.to_csv("/kaggle/working/xgb_7stat_experiment_summary.csv", index=False)

EXPERIMENT SUMMARY — XGBoost on 7-stat aggregated eGeMAPS
Feature vector length     : 161  (23 eGeMAPS features x 7 stats)
Train / Dev / Test counts : 163 / 56 / 56
Train class distribution  : {0: 126, 1: 37}
Dev class distribution    : {0: 44, 1: 12}
Test class distribution   : {0: 39, 1: 17}
scale_pos_weight (train)  : 3.405
Best hyperparameters      : {'max_depth': 2, 'n_estimators': 200, 'learning_rate': 0.03, 'subsample': 0.8, 'colsample_bytree': 0.6}
Best DEV threshold        : 0.57
----------------------------------------------------------------------
Split  Accuracy  Precision  Recall     F1  ROC_AUC  PR_AUC
  DEV    0.8036     0.5714  0.3333 0.4211   0.5095  0.3472
 TEST    0.7500     0.5882  0.5882 0.5882   0.7466  0.6232
----------------------------------------------------------------------
DEV  confusion matrix [[TN FP] [FN TP]]: [[41, 3], [8, 4]]
TEST confusion matrix [[TN FP] [FN TP]]: [[32, 7], [7, 10]]


In [7]:
import pandas as pd
from pathlib import Path

DATASET_ROOT = Path(
    "/kaggle/input/datasets/am2008/edaic-full-data/edaic_btp_dataset"
)

sample = next(
    (DATASET_ROOT / "raw_egemaps").glob("*_P/*egemaps.csv")
)

df = pd.read_csv(sample, sep=";")

feature_cols = [
    c for c in df.columns
    if c not in ["name", "frameTime"]
]

print("File:", sample)
print("Number of features:", len(feature_cols))
print("\nFeatures:")
for i, col in enumerate(feature_cols, 1):
    print(i, col)

File: /kaggle/input/datasets/am2008/edaic-full-data/edaic_btp_dataset/raw_egemaps/335_P/335_OpenSMILE2.3.0_egemaps.csv
Number of features: 23

Features:
1 Loudness_sma3
2 alphaRatio_sma3
3 hammarbergIndex_sma3
4 slope0-500_sma3
5 slope500-1500_sma3
6 spectralFlux_sma3
7 mfcc1_sma3
8 mfcc2_sma3
9 mfcc3_sma3
10 mfcc4_sma3
11 F0semitoneFrom27.5Hz_sma3nz
12 jitterLocal_sma3nz
13 shimmerLocaldB_sma3nz
14 HNRdBACF_sma3nz
15 logRelF0-H1-H2_sma3nz
16 logRelF0-H1-A3_sma3nz
17 F1frequency_sma3nz
18 F1bandwidth_sma3nz
19 F1amplitudeLogRelF0_sma3nz
20 F2frequency_sma3nz
21 F2amplitudeLogRelF0_sma3nz
22 F3frequency_sma3nz
23 F3amplitudeLogRelF0_sma3nz
